In [ ]:
import argparse
import datetime as dt
import json
import logging
import os
from pathlib import Path
from typing import Optional
import time

import numpy as np
import safetensors.torch
import torch
import torchaudio
from huggingface_hub import hf_hub_download
from lhotse.utils import fix_random_seed
from vocos import Vocos

from zipvoice.models.zipvoice import ZipVoice
from zipvoice.models.zipvoice_distill import ZipVoiceDistill
from zipvoice.tokenizer.tokenizer import (
    EmiliaTokenizer,
    EspeakTokenizer,
    LibriTTSTokenizer,
    SimpleTokenizer,
)
from zipvoice.utils.checkpoint import load_checkpoint
from zipvoice.utils.common import AttributeDict
from zipvoice.utils.feature import VocosFbank

In [ ]:
HUGGINGFACE_REPO = "k2-fsa/ZipVoice"
MODEL_DIR = {
    "zipvoice": "zipvoice",
    "zipvoice_distill": "zipvoice_distill",
}

def get_vocoder(vocos_local_path: Optional[str] = None):
    if vocos_local_path:
        vocoder = Vocos.from_hparams(f"{vocos_local_path}/config.yaml")
        state_dict = torch.load(
            f"{vocos_local_path}/pytorch_model.bin",
            weights_only=True,
            map_location="cpu",
        )
        vocoder.load_state_dict(state_dict)
    else:
        vocoder = Vocos.from_pretrained("charactr/vocos-mel-24khz")
    return vocoder

In [ ]:
from argparse import Namespace

default_args = {
    "model_name": "zipvoice_distill",
    "model_dir": None,
    "checkpoint_name": "model.pt",
    "vocoder_path": None,
    "target_rms": 0.1,
    "tokenizer": "emilia",
    "lang": "en-us",
    "test_list": None,
    "prompt_wav": None,
    "prompt_text": None,
    "text": None,
    "res_dir": "results",
    "res_wav_path": "result.wav",
    "guidance_scale": None,
    "num_step": None,
    "feat_scale": 0.1,
    "speed": 1.0,
    "seed": 111,
    "t_shift": 0.5
}
params = Namespace(**default_args)
params.prompt_wav = "/workspace/sonussocket/output.wav"
params.prompt_text = "This is a test."
params.text = "Tell me anything just do it!!!"

fix_random_seed(params.seed)

model_defaults = {
    "zipvoice": {
        "num_step": 16,
        "guidance_scale": 1.0,
    },
    "zipvoice_distill": {
        "num_step": 8,
        "guidance_scale": 3.0,
    },
}

model_specific_defaults = model_defaults.get(params.model_name, {})

for param, value in model_specific_defaults.items():
    if getattr(params, param) is None:
        setattr(params, param, value)
        logging.info(f"Setting {param} to default value: {value}")

assert (params.test_list is not None) ^ (
    (params.prompt_wav and params.prompt_text and params.text) is not None
), (
    "For inference, please provide prompts and text with either '--test-list'"
    " or '--prompt-wav, --prompt-text and --text'."
)

if params.model_dir is not None:
    params.model_dir = Path(params.model_dir)
    if not params.model_dir.is_dir():
        raise FileNotFoundError(f"{params.model_dir} does not exist")
    for filename in [params.checkpoint_name, "model.json", "tokens.txt"]:
        if not (params.model_dir / filename).is_file():
            raise FileNotFoundError(f"{params.model_dir / filename} does not exist")
    model_ckpt = params.model_dir / params.checkpoint_name
    model_config = params.model_dir / "model.json"
    token_file = params.model_dir / "tokens.txt"
    logging.info(
        f"Using local model dir {params.model_dir}, "
        f"checkpoint {params.checkpoint_name}"
    )
else:
    logging.info("Using pretrained model from the huggingface")
    logging.info("Downloading the requires files from HuggingFace")
    model_ckpt = hf_hub_download(
        HUGGINGFACE_REPO, filename=f"{MODEL_DIR[params.model_name]}/model.pt"
    )
    model_config = hf_hub_download(
        HUGGINGFACE_REPO, filename=f"{MODEL_DIR[params.model_name]}/model.json"
    )

    token_file = hf_hub_download(
        HUGGINGFACE_REPO, filename=f"{MODEL_DIR[params.model_name]}/tokens.txt"
    )

logging.info("Loading model...")

if params.tokenizer == "emilia":
    tokenizer = EmiliaTokenizer(token_file=token_file)
elif params.tokenizer == "libritts":
    tokenizer = LibriTTSTokenizer(token_file=token_file)
elif params.tokenizer == "espeak":
    tokenizer = EspeakTokenizer(token_file=token_file, lang=params.lang)
else:
    assert params.tokenizer == "simple"
    tokenizer = SimpleTokenizer(token_file=token_file)

tokenizer_config = {"vocab_size": tokenizer.vocab_size, "pad_id": tokenizer.pad_id}

with open(model_config, "r") as f:
    model_config = json.load(f)

if params.model_name == "zipvoice":
    model = ZipVoice(
        **model_config["model"],
        **tokenizer_config,
    )
else:
    assert params.model_name == "zipvoice_distill"
    model = ZipVoiceDistill(
        **model_config["model"],
        **tokenizer_config,
    )

if str(model_ckpt).endswith(".safetensors"):
    safetensors.torch.load_model(model, model_ckpt)
elif str(model_ckpt).endswith(".pt"):
    load_checkpoint(filename=model_ckpt, model=model, strict=True)
else:
    raise NotImplementedError(f"Unsupported model checkpoint format: {model_ckpt}")

if torch.cuda.is_available():
    params.device = torch.device("cuda", 0)
elif torch.backends.mps.is_available():
    params.device = torch.device("mps")
else:
    params.device = torch.device("cpu")
logging.info(f"Device: {params.device}")

model = model.to(params.device)
model.eval()

vocoder = get_vocoder(params.vocoder_path)
vocoder = vocoder.to(params.device)
vocoder.eval()

if model_config["feature"]["type"] == "vocos":
    feature_extractor = VocosFbank()
else:
    raise NotImplementedError(
        f"Unsupported feature type: {model_config['feature']['type']}"
    )

params.sampling_rate = model_config["feature"]["sampling_rate"]

In [ ]:
def generate_sentence(
    save_path: str,
    prompt_text: str,
    prompt_wav: str,
    text: str,
    model: torch.nn.Module,
    vocoder: torch.nn.Module,
    tokenizer: EmiliaTokenizer,
    feature_extractor: VocosFbank,
    device: torch.device,
    num_step: int = 16,
    guidance_scale: float = 1.0,
    speed: float = 1.0,
    t_shift: float = 0.5,
    target_rms: float = 0.1,
    feat_scale: float = 0.1,
    sampling_rate: int = 24000,
):
    st = time.time()

    tokens = tokenizer.texts_to_token_ids([text])
    prompt_tokens = tokenizer.texts_to_token_ids([prompt_text])

    # Load and preprocess prompt wav
    prompt_wav, prompt_sampling_rate = torchaudio.load(prompt_wav)

    if prompt_sampling_rate != sampling_rate:
        resampler = torchaudio.transforms.Resample(
            orig_freq=prompt_sampling_rate, new_freq=sampling_rate
        )
        prompt_wav = resampler(prompt_wav)

    prompt_rms = torch.sqrt(torch.mean(torch.square(prompt_wav)))
    if prompt_rms < target_rms:
        prompt_wav = prompt_wav * target_rms / prompt_rms

    # Extract features from prompt wav
    prompt_features = feature_extractor.extract(
        prompt_wav, sampling_rate=sampling_rate
    ).to(device)

    prompt_features = prompt_features.unsqueeze(0) * feat_scale
    prompt_features_lens = torch.tensor([prompt_features.size(1)], device=device)

    set_time = time.time() - st

    # Start timing
    start_t = dt.datetime.now() # 약 5%

    # Generate features
    (
        pred_features,
        pred_features_lens,
        pred_prompt_features,
        pred_prompt_features_lens,
    ) = model.sample(
        tokens=tokens,
        prompt_tokens=prompt_tokens,
        prompt_features=prompt_features,
        prompt_features_lens=prompt_features_lens,
        speed=speed,
        t_shift=t_shift,
        duration="predict",
        num_step=num_step,
        guidance_scale=guidance_scale,
    )

    # Postprocess predicted features
    pred_features = pred_features.permute(0, 2, 1) / feat_scale  # (B, C, T)

    # Start vocoder processing
    start_vocoder_t = dt.datetime.now()
    wav = vocoder.decode(pred_features).squeeze(1).clamp(-1, 1)

    # Calculate processing times and real-time factors
    t = (dt.datetime.now() - start_t).total_seconds()
    t_no_vocoder = (start_vocoder_t - start_t).total_seconds()
    t_vocoder = (dt.datetime.now() - start_vocoder_t).total_seconds()
    wav_seconds = wav.shape[-1] / sampling_rate
    rtf = t / wav_seconds
    rtf_no_vocoder = t_no_vocoder / wav_seconds
    rtf_vocoder = t_vocoder / wav_seconds
    metrics = {
        "t": t,
        "t_no_vocoder": t_no_vocoder,
        "t_vocoder": t_vocoder,
        "wav_seconds": wav_seconds,
        "rtf": rtf,
        "rtf_no_vocoder": rtf_no_vocoder,
        "rtf_vocoder": rtf_vocoder,
        "set_time": set_time
    }

    # Adjust wav volume if necessary
    if prompt_rms < target_rms:
        wav = wav * prompt_rms / target_rms
    # torchaudio.save(save_path, wav.cpu(), sample_rate=sampling_rate)

    return wav.cpu(), metrics

In [ ]:
waveform, metrics = generate_sentence(
    save_path=params.res_wav_path,
    prompt_text=params.prompt_text,
    prompt_wav=params.prompt_wav,
    text=params.text,
    model=model,
    vocoder=vocoder,
    tokenizer=tokenizer,
    feature_extractor=feature_extractor,
    device=params.device,
    num_step=params.num_step,
    guidance_scale=params.guidance_scale,
    speed=params.speed,
    t_shift=params.t_shift,
    target_rms=params.target_rms,
    feat_scale=params.feat_scale,
    sampling_rate=params.sampling_rate,
)

In [ ]:
params

In [ ]:
from IPython.display import Audio

display(Audio(waveform, rate=24000))

In [1]:
from tts.zipvoice_infer import load_infer_context, generate_sentence

ctx = load_infer_context()

/usr/local/lib/python3.11/dist-packages/vocos/pretrained.py:70: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(model_path, map_location="cpu")


In [11]:
prompt_wav_path = "/workspace/sonussocket/iamthetext.wav"
prompt_text = "I am the text to be synthesized."
text_chunk = "Tell me anything just do it!!!"

import time

start_time = time.time()
wav, info = generate_sentence(
    prompt_text=prompt_text,
    prompt_wav_path=prompt_wav_path,
    text=text_chunk,
    ctx=ctx,
)
print(f"[{wav.shape[-1]/24000:2f}] - {time.time() - start_time:.3f}")

[2.752000] - 0.481


In [10]:
from IPython.display import Audio

display(Audio(wav, rate=24000))

In [16]:
prompt_wav_path = "/workspace/sonussocket/iamthetext.wav"
prompt_text = "I am the text to be synthesized."
import time


texts = [
    "Did you hear that?",
    "Did you hear that? I think it came from the kitchen. Wow, that’s incredible!",
    "Did you hear that? I think it came from the kitchen. Wow, that’s incredible! I didn’t expect it at all.",
    "Did you hear that? I think it came from the kitchen. Wow, that’s incredible! I didn’t expect it at all. Excuse me, could you tell me where the station is?",
    "Did you hear that? I think it came from the kitchen. Wow, that’s incredible! I didn’t expect it at all. Excuse me, could you tell me where the station is? Hold on—are you serious right now?",
    "Did you hear that? I think it came from the kitchen. Wow, that’s incredible! I didn’t expect it at all. Excuse me, could you tell me where the station is? Hold on—are you serious right now? That’s too expensive. Maybe we should wait.",
]

for text_chunk in texts:
    start_time = time.time()
    wav, info = generate_sentence(
        prompt_text=prompt_text,
        prompt_wav_path=prompt_wav_path,
        text=text_chunk,
        ctx=ctx,
    )
    print(f"[{wav.shape[-1]/24000:2f}] - {time.time() - start_time:.3f}")

[1.536000] - 0.482
[6.080000] - 0.452
[8.512000] - 0.484
[12.885333] - 0.599
[15.968000] - 0.645
[19.530667] - 0.662


In [5]:
import time
from IPython.display import Audio

prompt_wav_path = "/workspace/sonussocket/elevenlabs3.mp3"
prompt_text = "In the ancient land of Eldoria, where skies shimmered and forests, whispered secrets to the wind, lived a dragon named Zephyros. Not the “burn it all down” kind... but he was gentle, wise, with eyes like old stars. Even the birds fell silent when he passed."
prompt_text = "Thank you for calling Tech Solutions. My name is Sarah, how can I help you today? Oh no, I'm really sorry to hear you're having trouble with your new device. That sounds frustrating. Okay, could you tell me a little more about what you're seeing on the screen?"

texts = [
    "No way! You actually did it?",
    "Stop right there—what do you think you’re doing?",
    "I can’t believe this… it’s unbelievable!",
    "Shhh, be quiet. Did you hear that noise?",
    "Come on, hurry up! We’re going to be late!",
    "Oh please, don’t tell me you forgot again.",
    "Wait, what?! That makes no sense at all.",
    "I told you already—I’m not going back there.",
    "Ha! That was the funniest thing I’ve ever seen.",
    "Ugh, I’m so tired… I just need one more coffee.",
    "Look out! That car is coming way too fast!",
    "Yes, yes, yes! Finally, it worked!",
    "Oh no… I really messed up this time.",
    "Are you kidding me right now?",
    "Listen carefully: this is the most important part.",
]

for text_chunk in texts:
    start_time = time.time()
    wav, info = generate_sentence(
        prompt_text=prompt_text,
        prompt_wav_path=prompt_wav_path,
        text=text_chunk,
        ctx=ctx,
        speed=0.6
    )
    print(text_chunk)
    print(f"[{wav.shape[-1]/24000:2f}] - {time.time() - start_time:.3f}")
    display(Audio(wav, rate=24000))


No way! You actually did it?
[1.162667] - 0.458


Stop right there—what do you think you’re doing?
[1.664000] - 0.444


I can’t believe this… it’s unbelievable!
[1.376000] - 0.500


Shhh, be quiet. Did you hear that noise?
[1.984000] - 0.692


Come on, hurry up! We’re going to be late!
[1.525333] - 0.445


...

Look out! That car is coming way too fast!
[1.632000] - 0.460


Yes, yes, yes! Finally, it worked!
[1.312000] - 0.449


Oh no… I really messed up this time.
[1.344000] - 0.399


Are you kidding me right now?
[1.098667] - 0.478


Listen carefully: this is the most important part.
[1.877333] - 0.445
